# keplerv — AI 3D reconstruction (Shap-E diffusion model)

Turns a single vehicle photo into a real 3D mesh using **Shap-E**, OpenAI's image-to-3D diffusion model (via Hugging Face `diffusers`). Exposes a small API over a public ngrok tunnel so keplerv.com can call it.

**Before you run:**
1. `Runtime` → `Change runtime type` → set Hardware accelerator to **T4 GPU** → Save.
2. Get a free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken (free account, no credit card) and paste it when cell 4 asks for it.
3. `Runtime` → `Run all`. First run downloads model weights (a few GB) — expect several minutes.
4. Copy the public URL printed by the last cell and paste it into keplerv.com's reconstruction settings (see repo README).

**This only works while this notebook keeps running.** Free Colab GPU sessions disconnect after a period of idle time or ~12h max — if the website's reconstruction feature stops working, this notebook probably disconnected and needs to be rerun.

In [ ]:
# 1. Install dependencies
!pip install -q diffusers transformers accelerate trimesh pyngrok fastapi "uvicorn[standard]" nest-asyncio python-multipart pillow requests

In [ ]:
# 2. Load the Shap-E image-to-3D diffusion pipeline (downloads weights on first run)
import torch
from diffusers import ShapEImg2ImgPipeline

assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > T4 GPU, then rerun."

pipe = ShapEImg2ImgPipeline.from_pretrained("openai/shap-e-img2img", torch_dtype=torch.float16)
pipe = pipe.to("cuda")
print("Shap-E loaded on", torch.cuda.get_device_name(0))

In [ ]:
# 3. Reconstruction function: image URL -> GLB mesh bytes
import io
import requests
from PIL import Image
import trimesh
import numpy as np

def reconstruct_to_glb(image_url: str, steps: int = 64, guidance_scale: float = 3.0) -> bytes:
    resp = requests.get(image_url, timeout=30)
    resp.raise_for_status()
    image = Image.open(io.BytesIO(resp.content)).convert("RGB")

    result = pipe(
        image=image,
        guidance_scale=guidance_scale,
        num_inference_steps=steps,
        output_type="mesh",
    )
    mesh_out = result.images[0]

    verts = mesh_out.verts.detach().cpu().numpy() if torch.is_tensor(mesh_out.verts) else np.asarray(mesh_out.verts)
    faces = mesh_out.faces.detach().cpu().numpy() if torch.is_tensor(mesh_out.faces) else np.asarray(mesh_out.faces)

    vertex_colors = None
    if getattr(mesh_out, "vertex_channels", None):
        try:
            rgb = np.stack(
                [mesh_out.vertex_channels[c].detach().cpu().numpy() for c in "RGB"], axis=1
            )
            vertex_colors = np.clip(rgb * 255, 0, 255).astype(np.uint8)
        except Exception as e:
            print("no vertex colors:", e)

    tri_mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_colors=vertex_colors, process=False)
    buf = io.BytesIO()
    tri_mesh.export(buf, file_type="glb")
    return buf.getvalue()

In [ ]:
# 4. ngrok auth — paste your free token from https://dashboard.ngrok.com/get-started/your-authtoken
from getpass import getpass
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = getpass("Paste your ngrok authtoken: ")
conf.get_default().auth_token = NGROK_AUTHTOKEN

In [ ]:
# 5. FastAPI server + public tunnel
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://www.keplerv.com", "https://keplerv.com", "http://localhost:4399"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ReconstructRequest(BaseModel):
    image_url: str
    steps: int | None = 64

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/reconstruct")
def reconstruct(req: ReconstructRequest):
    try:
        glb_bytes = reconstruct_to_glb(req.image_url, steps=req.steps or 64)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    return Response(content=glb_bytes, media_type="model/gltf-binary")

nest_asyncio.apply()
public_url = ngrok.connect(8000, "http")
print("\n\u2705 Public API URL (paste into keplerv.com):", public_url)
print("   Health check:", str(public_url) + "/health\n")

uvicorn.run(app, host="0.0.0.0", port=8000)

### Troubleshooting
- **`assert torch.cuda.is_available()` fails** — you didn't enable the T4 GPU runtime (see step 1 above).
- **ngrok "authentication failed"** — token wasn't pasted correctly in cell 4; rerun that cell.
- **Cell 5 keeps running forever with no error** — that's correct! It's the live server. Leave it running; that's what keeps the website's 3D reconstruction feature alive.
- **Reconstruction looks blobby/low-detail** — this is expected. Shap-E is a fast, lightweight diffusion model; it produces a rough 3D shape + color, not a photorealistic scan. Salvage-vehicle photos (background clutter, damage, non-studio lighting) are also a hard case for it.